# One Grover Iteration for PRINCE Key Search

This notebook builds one Grover iteration for a known plaintext/ciphertext PRINCE key-search attack. The searched key is the real 128-bit key `k0 || k1`; `k0'` is computed reversibly from `k0`, used in the PRINCE encryption oracle, and then uncomputed.

The circuit structure is:

1. Put the 128 key qubits in superposition.
2. Compute `k0'` from `k0`.
3. Compute `Enc_K(plaintext)` into the state register.
4. Phase-mark keys where the computed ciphertext equals the target ciphertext.
5. Uncompute PRINCE and `k0'` so only the key phase remains.
6. Apply the 128-qubit Grover diffuser on the key register.


In [1]:
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit.library import RCCXGate

from Helpers import xor_constant_into_register, xor_register_into_register
from Sbox import quantum_sbox_nibble, quantum_sbox_inverse_nibble
from Mbox import quantum_m_layer, quantum_m_layer_inv


In [2]:
ROUND_CONSTANTS = [
    0x0000000000000000,
    0x13198A2E03707344,
    0xA4093822299F31D0,
    0x082EFA98EC4E6C89,
    0x452821E638D01377,
    0xBE5466CF34E90C6C,
    0x7EF84F78FD955CB1,
    0x85840851F1AC43AA,
    0xC882D32F25323C54,
    0x64A51195E0E3610D,
    0xD3B5A399CA0C2399,
    0xC0AC29B7C97C50DD,
]

RC0 = ROUND_CONSTANTS[0]
MASK64 = (1 << 64) - 1


## Efficient PRINCE Core

In [3]:
def efficient_s_layer(qc, state, anc):
    for i in range(16):
        nibble = [state[4 * i + k] for k in range(4)]
        scratch = [anc[14 * i + k] for k in range(14)]
        quantum_sbox_nibble(qc, nibble, scratch)


def efficient_s_layer_inv(qc, state, anc):
    for i in range(16):
        nibble = [state[4 * i + k] for k in range(4)]
        scratch = [anc[14 * i + k] for k in range(14)]
        quantum_sbox_inverse_nibble(qc, nibble, scratch)


def efficient_round(qc, state, anc, qk1, rc):
    efficient_s_layer(qc, state, anc)
    quantum_m_layer(qc, state)
    xor_constant_into_register(qc, state, rc)
    xor_register_into_register(qc, qk1, state)


def efficient_inverse_round(qc, state, anc, qk1, rc):
    xor_register_into_register(qc, qk1, state)
    xor_constant_into_register(qc, state, rc)
    quantum_m_layer_inv(qc, state)
    efficient_s_layer_inv(qc, state, anc)


def append_prince_encryption(qc, state, anc, qk0, qk1, qk0p):
    xor_register_into_register(qc, qk0, state)
    xor_register_into_register(qc, qk1, state)
    xor_constant_into_register(qc, state, RC0)

    for i in range(1, 6):
        efficient_round(qc, state, anc, qk1, ROUND_CONSTANTS[i])

    efficient_s_layer(qc, state, anc)
    quantum_m_layer(qc, state)
    efficient_s_layer_inv(qc, state, anc)

    for i in range(6, 11):
        efficient_inverse_round(qc, state, anc, qk1, ROUND_CONSTANTS[i])

    xor_constant_into_register(qc, state, ROUND_CONSTANTS[11])
    xor_register_into_register(qc, qk1, state)
    xor_register_into_register(qc, qk0p, state)


def append_prince_encryption_inverse(qc, state, anc, qk0, qk1, qk0p):
    xor_register_into_register(qc, qk0p, state)
    xor_register_into_register(qc, qk1, state)
    xor_constant_into_register(qc, state, ROUND_CONSTANTS[11])

    for i in range(10, 5, -1):
        efficient_round(qc, state, anc, qk1, ROUND_CONSTANTS[i])

    efficient_s_layer(qc, state, anc)
    quantum_m_layer_inv(qc, state)
    efficient_s_layer_inv(qc, state, anc)

    for i in range(5, 0, -1):
        efficient_inverse_round(qc, state, anc, qk1, ROUND_CONSTANTS[i])

    xor_constant_into_register(qc, state, RC0)
    xor_register_into_register(qc, qk1, state)
    xor_register_into_register(qc, qk0, state)


def prince_encryption_subcircuit():
    state = QuantumRegister(64, name="state")
    anc = QuantumRegister(16 * 14, name="anc")
    qk0 = QuantumRegister(64, name="k0")
    qk1 = QuantumRegister(64, name="k1")
    qk0p = QuantumRegister(64, name="k0p")
    qc = QuantumCircuit(state, anc, qk0, qk1, qk0p, name="PRINCE")
    append_prince_encryption(qc, state, anc, qk0, qk1, qk0p)
    return qc


## Grover Building Blocks

In [4]:
def compute_k0_prime(qc, qk0, qk0p):
    # k0_prime = ROR64(k0, 1) xor (k0 >> 63).
    for out_bit in range(64):
        qc.cx(qk0[(out_bit + 1) % 64], qk0p[out_bit])
    qc.cx(qk0[63], qk0p[0])


def uncompute_k0_prime(qc, qk0, qk0p):
    # Same CNOT network reverses itself.
    qc.cx(qk0[63], qk0p[0])
    for out_bit in range(63, -1, -1):
        qc.cx(qk0[(out_bit + 1) % 64], qk0p[out_bit])


def multi_control_phase_on_ones(qc, controls, scratch):
    if len(controls) == 1:
        qc.z(controls[0])
        return
    if len(scratch) < len(controls) - 1:
        raise ValueError("Need len(controls)-1 clean scratch qubits")

    qc.ccx(controls[0], controls[1], scratch[0])
    for i in range(2, len(controls)):
        qc.ccx(scratch[i - 2], controls[i], scratch[i - 1])

    qc.z(scratch[len(controls) - 2])

    for i in range(len(controls) - 1, 1, -1):
        qc.ccx(scratch[i - 2], controls[i], scratch[i - 1])
    qc.ccx(controls[0], controls[1], scratch[0])


def phase_mark_ciphertext(qc, state, target_ciphertext, scratch):
    # Convert equality Enc_K(pt) == ct into all-ones controls.
    xor_constant_into_register(qc, state, target_ciphertext)
    for bit in range(64):
        qc.x(state[bit])

    multi_control_phase_on_ones(qc, list(state), scratch[:63])

    for bit in range(64):
        qc.x(state[bit])
    xor_constant_into_register(qc, state, target_ciphertext)


def diffuser_128(qc, key_qubits, scratch):
    for q in key_qubits:
        qc.h(q)
        qc.x(q)

    multi_control_phase_on_ones(qc, key_qubits, scratch[:127])

    for q in key_qubits:
        qc.x(q)
        qc.h(q)


## One Iteration Circuit

In [5]:
def build_one_grover_iteration(plaintext_64bit, target_ciphertext_64bit):
    state = QuantumRegister(64, name="state")
    qk0 = QuantumRegister(64, name="k0")
    qk1 = QuantumRegister(64, name="k1")
    qk0p = QuantumRegister(64, name="k0p")
    sbox_anc = QuantumRegister(16 * 14, name="sbox_anc")
    phase_anc = QuantumRegister(127, name="phase_anc")

    qc = QuantumCircuit(state, qk0, qk1, qk0p, sbox_anc, phase_anc)

    for bit in range(64):
        if (plaintext_64bit >> bit) & 1:
            qc.x(state[bit])

    key_qubits = list(qk0) + list(qk1)
    for q in key_qubits:
        qc.h(q)

    compute_k0_prime(qc, qk0, qk0p)

    append_prince_encryption(qc, state, sbox_anc, qk0, qk1, qk0p)

    phase_mark_ciphertext(qc, state, target_ciphertext_64bit, list(phase_anc))

    append_prince_encryption_inverse(qc, state, sbox_anc, qk0, qk1, qk0p)
    uncompute_k0_prime(qc, qk0, qk0p)

    diffuser_128(qc, key_qubits, list(phase_anc))
    return qc


## Resource Counts for One Iteration

In [6]:
IGNORED_OPS = {"barrier", "measure"}
RAW_BASIS = ["h", "x", "z", "cx", "ccx"]
CLIFFORD_T_BASIS = ["h", "x", "z", "s", "sdg", "cx", "t", "tdg"]
CLIFFORD_OPS = {"h", "x", "z", "s", "sdg", "cx"}
T_OPS = {"t", "tdg"}


def clean_counts(qc):
    return {k: v for k, v in dict(qc.count_ops()).items() if k not in IGNORED_OPS}


def clean_depth(qc):
    return qc.depth(filter_function=lambda inst: inst.operation.name not in IGNORED_OPS)


def replace_toffoli_with_relative_phase(qc):
    converted = QuantumCircuit(*qc.qregs, *qc.cregs)
    for circuit_instruction in qc.data:
        op = circuit_instruction.operation
        if op.name == "ccx":
            converted.append(RCCXGate(), circuit_instruction.qubits, circuit_instruction.clbits)
        else:
            converted.append(op, circuit_instruction.qubits, circuit_instruction.clbits)
    return converted


def exact_resources(qc):
    raw = transpile(qc, basis_gates=RAW_BASIS, optimization_level=3)
    raw_ops = clean_counts(raw)

    clifford_t = transpile(qc, basis_gates=CLIFFORD_T_BASIS, optimization_level=3)
    ct_ops = clean_counts(clifford_t)

    return {
        "raw": {
            "total_qubits": raw.num_qubits,
            "h_gates": raw_ops.get("h", 0),
            "x_gates": raw_ops.get("x", 0),
            "z_gates": raw_ops.get("z", 0),
            "cx_gates": raw_ops.get("cx", 0),
            "toffoli_gates": raw_ops.get("ccx", 0),
            "total_gates": sum(raw_ops.values()),
            "depth": clean_depth(raw),
        },
        "clifford_t": {
            "total_qubits": clifford_t.num_qubits,
            "total_clifford_gates": sum(ct_ops.get(name, 0) for name in CLIFFORD_OPS),
            "total_t_gates": sum(ct_ops.get(name, 0) for name in T_OPS),
            "t_depth": clifford_t.depth(filter_function=lambda inst: inst.operation.name in T_OPS),
            "total_depth": clean_depth(clifford_t),
            "gate_wise_counts": ct_ops,
        },
    }


def aggressive_resources(qc):
    relative = replace_toffoli_with_relative_phase(qc)
    clifford_t = transpile(relative, basis_gates=CLIFFORD_T_BASIS, optimization_level=0)
    ct_ops = clean_counts(clifford_t)
    return {
        "total_qubits": clifford_t.num_qubits,
        "total_clifford_gates": sum(ct_ops.get(name, 0) for name in CLIFFORD_OPS),
        "total_t_gates": sum(ct_ops.get(name, 0) for name in T_OPS),
        "t_depth": clifford_t.depth(filter_function=lambda inst: inst.operation.name in T_OPS),
        "total_depth": clean_depth(clifford_t),
        "gate_wise_counts": ct_ops,
    }


plaintext = 0x1111111111111111
target_ciphertext = 0x44FBFF21F4BA60E1
grover_one = build_one_grover_iteration(plaintext, target_ciphertext)

one_iteration_exact = exact_resources(grover_one)
one_iteration_aggressive = aggressive_resources(grover_one)

print("One Grover iteration, exact raw:")
print(one_iteration_exact["raw"])
print("\nOne Grover iteration, exact Clifford+T:")
print({k: v for k, v in one_iteration_exact["clifford_t"].items() if k != "gate_wise_counts"})
print("\nOne Grover iteration, aggressive relative-phase Clifford+T:")
print({k: v for k, v in one_iteration_aggressive.items() if k != "gate_wise_counts"})


One Grover iteration, exact raw:
{'total_qubits': 607, 'h_gates': 384, 'x_gates': 2710, 'z_gates': 2, 'cx_gates': 27970, 'toffoli_gates': 15740, 'total_gates': 46806, 'depth': 2015}

One Grover iteration, exact Clifford+T:
{'total_qubits': 607, 'total_clifford_gates': 141822, 'total_t_gates': 110180, 't_depth': 4452, 'total_depth': 11965}

One Grover iteration, aggressive relative-phase Clifford+T:
{'total_qubits': 607, 'total_clifford_gates': 94782, 'total_t_gates': 47790, 't_depth': 2010, 'total_depth': 5617}
